<a href="https://colab.research.google.com/github/Ben-3id/Fine-Tune/blob/main/FineTune.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch

print("GPU:", torch.cuda.get_device_name(0))
print("CUDA available:", torch.cuda.is_available())

AssertionError: Torch not compiled with CUDA enabled

In [ ]:
!pip uninstall -y torchao

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


In [ ]:
!pip install transformers datasets peft  trl accelerate torchao > 0.16.0

#Load DataSet

In [ ]:
from datasets import load_dataset

In [ ]:
data = load_dataset(path="MBZUAI-Paris/Egyptian-SFT-Mixture" , split=["train" , "test"])

README.md:   0%|          | 0.00/3.17k [00:00<?, ?B/s]

data/train-00000-of-00010.parquet: reconstructing file:   0%|          |  0.00B /  189MB            

data/train-00000-of-00010.parquet: downloading bytes:           |  0.00B            

data/train-00001-of-00010.parquet: reconstructing file:   0%|          |  0.00B /  190MB            

data/train-00001-of-00010.parquet: downloading bytes:           |  0.00B            

...

data/train-00009-of-00010.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 10.5MB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/1817288 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/35838 [00:00<?, ? examples/s]

In [ ]:
data

[Dataset({
     features: ['messages', 'dataset', 'id', 'length_tokens'],
     num_rows: 1817288
 }),
 Dataset({
     features: ['messages', 'dataset', 'id', 'length_tokens'],
     num_rows: 35838
 })]

In [ ]:
train_data = data[0].shuffle(seed=42).select(range(50000))
val_test_data =  data[1].train_test_split(test_size=0.5)
val_data = val_test_data['train']
test_data = val_test_data['test']

In [ ]:
print(f"training data ---> {train_data}")
print(f"validating data ---> {val_data}")
print(f"test data ---> {test_data}")

training data ---> Dataset({
    features: ['messages', 'dataset', 'id', 'length_tokens'],
    num_rows: 50000
})
validating data ---> Dataset({
    features: ['messages', 'dataset', 'id', 'length_tokens'],
    num_rows: 17919
})
test data ---> Dataset({
    features: ['messages', 'dataset', 'id', 'length_tokens'],
    num_rows: 17919
})


#Init Model

In [ ]:
from transformers import (
    AutoTokenizer ,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    Pipeline
    )

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B")
model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-0.5B")

config.json:   0%|          | 0.00/681 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.23k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

#LoRA Config

In [ ]:
from peft import (LoraConfig,
                  TaskType,
                  get_peft_model)

In [ ]:
config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=4,
    target_modules =[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
    ]
)

In [ ]:
injed_model = get_peft_model(model, config)

injed_model.print_trainable_parameters()

trainable params: 540,672 || all params: 494,573,440 || trainable%: 0.1093


In [ ]:
print(model)

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 896)
    (layers): ModuleList(
      (0-23): 24 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): lora.Linear(
            (base_layer): Linear(in_features=896, out_features=896, bias=True)
            (lora_dropout): ModuleDict(
              (default): Identity()
            )
            (lora_A): ModuleDict(
              (default): Linear(in_features=896, out_features=4, bias=False)
            )
            (lora_B): ModuleDict(
              (default): Linear(in_features=4, out_features=896, bias=False)
            )
            (lora_embedding_A): ParameterDict()
            (lora_embedding_B): ParameterDict()
            (lora_magnitude_vector): ModuleDict()
          )
          (k_proj): lora.Linear(
            (base_layer): Linear(in_features=896, out_features=128, bias=True)
            (lora_dropout): ModuleDict(
              (default): Identity()
            )
   

In [ ]:
for name , module in model.named_modules():
  print(f"{name} ---> {module}")

# Training

In [ ]:
from trl import SFTTrainer , SFTConfig

In [ ]:
training_arg = SFTConfig(
    output_dir = "finetune-result",
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    save_steps=100,
    logging_steps=100,
    val
    save_strategy='steps',
    fp16=True,
    bf16=False,
)

trainer = SFTTrainer(
    model=injed_model,
    args=training_arg,
    train_dataset=train_data,
    eval_dataset=val_data,
    processing_class=tokenizer,
)

Tokenizing train dataset:   0%|          | 0/50000 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/50000 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/50000 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/50000 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/17919 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/17919 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/17919 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/17919 [00:00<?, ? examples/s]

In [ ]:
trainer.train()

Step,Training Loss
100,2.956133
200,2.845536
300,2.715996
400,2.629495
500,2.676396
600,2.641518
700,2.596908
800,2.528130
900,2.539750
1000,2.486616
